#### 통합본 구조
1. 카카오맵 리뷰 링크 입력 --> ① 장소리뷰 텍스트 ② 장소리뷰 날짜 + 블로그리뷰 날짜 추출
2. 형태소분석 --> score_morph 계산
3. 감성분석 --> score_sentiment 계산
4, 시간성분석 --> score_time 계산
5. score_total 계산 --> threshold 이상이면 Viral, 미만이면 Non viral Return

#### 참고사항
1. CPU 상황 가정
2. 실행시간 길지 않도록 가능한 병렬처리

### 추가 처리 필요사항
1. 패키지 미리 설치해놓고 로드만 하도록

---

#### Part1
수정사항
1. wait = WebDriverWait(driver, 10) --> wait = WebDriverWait(driver, 5)

In [3]:

def collect_kakao_reviews():
  # Install libraries
  !pip install selenium
  !apt-get update
  !apt install chromium-chromedriver
  !pip install fake_useragent
  !pip install deep-translator

  # Load libraries
  import sys
  sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
  from fake_useragent import UserAgent

  import time
  from selenium import webdriver
  from selenium.webdriver.common.by import By
  import pandas as pd
  import numpy as np
  from selenium.webdriver.support.ui import WebDriverWait
  from selenium.webdriver.support import expected_conditions as EC

  from bs4 import BeautifulSoup

  from deep_translator import GoogleTranslator

  # Set up Chrome options
  max_clicks = 10
  ua = UserAgent()
  chrome_options = webdriver.ChromeOptions()
  chrome_options.add_argument('--headless')
  chrome_options.add_argument('--no-sandbox')
  chrome_options.add_argument('--user-agent--="%s"' % str(ua.random))
  driver = webdriver.Chrome(options=chrome_options)

  # Get Kakao Map place URL from user input
  link = input("리뷰를 수집할 카카오맵 장소 URL을 입력하세요: ")
  driver.get(link)
  time.sleep(3)

  count = 0
  results = []
  stars = []
  wait = WebDriverWait(driver, 10)

  # Load all reviews by clicking 'More' button
  try:
      total_comments_element = wait.until(
          EC.presence_of_element_located((By.CSS_SELECTOR, "a.link_evaluation[data-target='comment'] .color_g")))
      total_comments_text = total_comments_element.text.strip()
      total_comments = int(total_comments_text.replace("(", "").replace(")", "").replace("개 후기", "").strip())
      print(f"후기 개수: {total_comments}")
  except Exception as e:
      print(f"후기 개수 추출 실패: {e}")
      total_comments = 0

  print("더보기 버튼 클릭 시작")
  count = 0
  last_review_count = 0

  while count < max_clicks:
      try:
          current_review_count = len(driver.find_elements(By.CSS_SELECTOR, "#mArticle > div.cont_evaluation > div.evaluation_review > ul > li"))
          if current_review_count == last_review_count:
              print("모든 리뷰를 로드했습니다.")
              break

          more_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#mArticle > div.cont_evaluation > div.evaluation_review > a")))
          driver.execute_script("arguments[0].click();", more_btn)
          count += 1
          time.sleep(3)

          last_review_count = current_review_count

      except Exception as e:
          print(f"더보기 버튼 클릭 종료: {e}")
          break

  print(f"더보기 버튼 총 {count}회 클릭 완료")

  # Extract reviews from the page source
  html = driver.page_source
  soup = BeautifulSoup(html, 'html.parser')
  review_elements = soup.select('#mArticle > div.cont_evaluation > div.evaluation_review > ul > li')

  for block in review_elements:
      star_tag = block.select_one('span.ico_star.inner_star')
      if star_tag and 'style' in star_tag.attrs:
          style_width = star_tag['style']
          width = int(style_width.split(":")[1].strip().strip(';').replace('%', ''))
          rating = width / 20
      else:
          print("별점 정보를 찾을 수 없습니다.")
          rating = None

      review_text = block.select_one('p.txt_comment > span').text.strip()
      review_date = block.select_one('span.time_write').text.strip() if block.select_one('span.time_write') else "날짜 없음"
      results.append({"date": review_date, "rating": rating, "review": review_text})

  # Translate reviews
  translated_results = []
  for trans_review in results:
      try:
          translated_review_text = GoogleTranslator(source='ko', target='en').translate(trans_review['review'])
      except Exception as e:
          translated_review_text = f"번역 실패: {str(e)}"
      translated_results.append({
          "date": trans_review["date"],
          "rating": trans_review["rating"],
          "original_review": trans_review["review"],
          "translated_review": translated_review_text
      })

  # Create DataFrame from translated results
  df = pd.DataFrame(translated_results, columns=["date", "rating", "original_review", "translated_review"])
  print(df)

  # Load all blog reviews by clicking 'More' button
  blog_results = []
  count_blog = 0
  while True:
      try:
          if count_blog > 100:
              print("더보기 버튼 100회 클릭 후 클릭 중단")
              break
          more_btn2 = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#mArticle > div.cont_review > div.wrap_list > a.link_more")))
          more_btn2.click()
          count_blog += 1
          time.sleep(3)
      except:
          print("더보기 버튼 총 ", count_blog, "회 클릭 완료")
          break
      blog_reviews = soup.select("#mArticle > div.cont_review > div.wrap_list > ul.list_review > li")
      for blog in blog_reviews:
          try:
              blog_title_tag = blog.select_one("strong.tit_story")
              blog_title = blog_title_tag.text.strip() if blog_title_tag else "블로그 제목 없음"

              blog_date_tag = blog.select_one("span.txt_date")
              blog_date = blog_date_tag.text.strip() if blog_date_tag else "날짜 없음"
          except:
              blog_title = "블로그 제목 없음"
              blog_date = "날짜 없음"
          blog_results.append({"date": blog_date, "name": blog_title})

  # Create DataFrame from blog results
  df2 = pd.DataFrame(list(blog_results), columns=["date", "name"])
  # print(df2)


  # Close the driver
  driver.quit()
  return df, df2

In [4]:
df1, df2 = collect_kakao_reviews()

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Fetched 384 kB in 1s (319 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading pac

In [6]:
df1.head() # 장소리뷰 날짜, 번역 전/후

,date,rating,original_review,translated_review
0,2024.11.29.,1.0,니 애미,Your mom
1,2024.11.17.,1.0,,
2,2024.11.15.,1.0,,
3,2024.11.07.,1.0,,
4,2024.11.06.,1.0,"사장이면서 문앞에서 삐끼역까지 하는 사람, 정말 최악의 장사꾼입니다. 불친절에 안하...",The person who is the boss and also acts as a ...


In [7]:
df2.head() # 블로그리뷰 날짜, 제목

,date,name
0,2024.11.16.,남산 돈가스 맛집 원조 남산 왕돈가스 주차 영업시간 메뉴 정보
1,2024.11.16.,"남산 돈까스?[원조 남산 왕돈까스]-남산 추천 맛집,남산 돈까스 맛집,남산 추억의 ..."
2,2024.11.11.,[남산타워 맛집] 원조남산왕돈까스
3,2024.11.10.,원조남산왕돈까스 l 남산돈까스는 여기가 원조!
4,2024.11.06.,"[서울 맛집] 남산을 간다면 역시 돈까스를 먹어야지! <원조> ""남산 돈까스"""


In [ ]:
def create_dataset(url, max_click = 10):
  # install libraries
  # !pip install selenium
  # !apt-get update
  # !apt install chromium-chromedriver
  # !pip install fake_useragent
  # !pip install deep-translator

  # load libraries
  import sys
  import pandas as pd
  import numpy as np
  import time

  from fake_useragent import UserAgent

  from selenium import webdriver # selenium
  from selenium import webdriver
  from selenium.webdriver.common.by import By
  from selenium.webdriver.support.ui import WebDriverWait
  from selenium.webdriver.support import expected_conditions as EC

  from bs4 import BeautifulSoup # beautifulsoup

  from deep_translator import GoogleTranslator # google translator

  # set libraries
  sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
  chrome_options = webdriver.ChromeOptions()
  chrome_options.add_argument('--headless')
  chrome_options.add_argument('--no-sandbox')
  chrome_options.add_argument('--user-agent--="%s"' % str(ua.random))
  driver = webdriver.Chrome(options=chrome_options)

  # set argument
  max_click = max_click
  url = url

  # connect to url
  ua = UserAgent()
  driver.get(url)
  time.sleep(3)

  # get location reviews and dates
  count = 0
